# Task 1.1 — Sentiment Analysis: Simple ANN + Bi-LSTM

- **Shared preprocessing pipeline** (`src.preprocessing`) — identical splits, vocab, and tokenisation as Task 1.2
- **TensorBoard** for training curves, LR, and gradient norms
- **Early stopping** to prevent overfitting
- **Label smoothing** on loss for both models
- **Model + vocab saved** to `./model/`

In [ ]:
import os
import sys
import time
import datetime
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, classification_report

sys.path.insert(0, os.path.join(os.getcwd(), "utils"))  # jupyter path fix
from utils.preprocessing import load_data, MAX_SEQ_LEN, PAD_IDX, MODEL_DIR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Device              : {device}")
print(f"Model output dir    : ./{MODEL_DIR}/")

Device              : cpu
Model output dir    : ./model/


## Model 1 — Simple ANN

Uses the shared `Vocabulary` + token IDs (same as LSTM/Transformer).

In [11]:
class SimpleANN(nn.Module):
    """
    Embedding bag → MLP classifier.
    Replaces TF-IDF with a learned embedding average so the same
    Vocabulary and DataLoaders can be shared with the Bi-LSTM and Transformer.
    """

    def __init__(self, vocab_size: int, embed_dim: int = 64, dropout: float = 0.4):
        super().__init__()
        self.embedding = nn.EmbeddingBag(
            vocab_size, embed_dim, padding_idx=PAD_IDX, mode="mean"
        )
        self.network = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, 2),   # 2-class logits → CrossEntropyLoss
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : (B, T) token ids
        embedded = self.embedding(x)  # (B, embed_dim)
        return self.network(embedded)  # (B, 2)


print("SimpleANN defined.")

SimpleANN defined.


## Model 2 — Bi-LSTM

In [12]:
class BiLSTMClassifier(nn.Module):
    """Bi-directional LSTM sentiment classifier."""

    def __init__(
        self,
        vocab_size:    int,
        embedding_dim: int   = 128,
        hidden_dim:    int   = 128,
        n_layers:      int   = 1,
        dropout:       float = 0.4,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(
            input_size    = embedding_dim,
            hidden_size   = hidden_dim,
            num_layers    = n_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if n_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 2)   # 2-class logits

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : (B, T)
        embedded = self.embedding(x)                             # (B, T, E)
        _, (hidden, _) = self.lstm(embedded)                     # hidden: (2, B, H)
        hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (B, 2H)
        return self.fc(self.dropout(hidden_cat))                 # (B, 2)


print("BiLSTMClassifier defined.")

BiLSTMClassifier defined.


## Shared training helpers

In [ ]:
def evaluate(model, loader):
    """Plain CE loss (no smoothing) so val loss is comparable across runs."""
    model.eval()
    criterion   = nn.CrossEntropyLoss()
    total_loss  = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y   = x.to(device), y.to(device)
            logits = model(x)
            total_loss += criterion(logits, y).item()
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    return (
        total_loss / len(loader),
        accuracy_score(all_labels, all_preds),
        all_preds,
        all_labels,
    )


def fit(
    model,
    train_loader,
    val_loader,
    model_name:              str   = "model",
    n_epochs:                int   = 30,
    lr:                      float = 1e-3,
    weight_decay:            float = 1e-2,
    label_smoothing:         float = 0.1,
    early_stopping_patience: int   = 8,
    model_dir:               str   = MODEL_DIR,
    log_dir:                 str   = "runs",
):
    """
    Generic training loop shared by SimpleANN and BiLSTMClassifier.

    Anti-overfitting knobs:
    - label_smoothing   : softens targets → less overconfident memorisation
    - weight_decay      : L2 via AdamW
    - early_stopping_patience : stops when val acc stagnates
    - dropout baked into both model architectures
    """
    run_name = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    writer   = SummaryWriter(log_dir=f"{log_dir}/{model_name}/{run_name}")
    print(f"[{model_name}] TensorBoard : {log_dir}/{model_name}/{run_name}")
    print(f"               Launch with : tensorboard --logdir {log_dir}")

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=lr * 0.01
    )

    best_val_acc = 0.0
    best_state   = None
    patience_ctr = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        t0         = time.time()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        train_loss               = epoch_loss / len(train_loader)
        val_loss, val_acc, _, _  = evaluate(model, val_loader)
        current_lr               = scheduler.get_last_lr()[0]
        elapsed                  = time.time() - t0

        writer.add_scalar("Loss/train",   train_loss, epoch)
        writer.add_scalar("Loss/val",     val_loss,   epoch)
        writer.add_scalar("Accuracy/val", val_acc,    epoch)
        writer.add_scalar("LR",           current_lr, epoch)
        writer.add_scalar("Grad_Norm",    grad_norm,  epoch)

        marker = " ◀ best" if val_acc > best_val_acc else ""
        print(
            f"Epoch {epoch:3d}/{n_epochs} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f} | Acc: {val_acc:.4f} | "
            f"{elapsed:.1f}s{marker}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= early_stopping_patience:
                print(
                    f"\n[early stop] No val acc improvement for "
                    f"{early_stopping_patience} epochs — stopping at epoch {epoch}."
                )
                break

    os.makedirs(model_dir, exist_ok=True)
    save_path = os.path.join(model_dir, f"{model_name}.pt")
    model.load_state_dict(best_state)
    torch.save({"model_state": best_state}, save_path)

    # Log hparams
    writer.add_hparams(
        {"lr": lr, "weight_decay": weight_decay, "label_smoothing": label_smoothing},
        {"hparam/best_val_acc": best_val_acc},
    )
    writer.close()

    print(f"[{model_name}] Saved → {save_path}")
    return model


print("Training helpers defined.")

Training helpers defined.


## Load data via shared pipeline

In [14]:
DATA_FILE  = "data/amazon_cells_labelled_LARGE_25K.txt"   # data/amazon_cells_labelled.txt
BATCH_SIZE = 64

train_loader, val_loader, test_loader, vocab = load_data(
    filepath   = DATA_FILE,
    batch_size = BATCH_SIZE,
    vocab_path = os.path.join(MODEL_DIR, "vocab.pkl"),  # shared with Task 1.2
)

print(f"Vocab size          : {len(vocab):,}")
print(f"Train batches       : {len(train_loader)}")
print(f"Val   batches       : {len(val_loader)}")
print(f"Test  batches       : {len(test_loader)}")

Vocab size          : 9,002
Train batches       : 313
Val   batches       : 40
Test  batches       : 40


## Train Simple ANN

In [15]:
ann_model = SimpleANN(vocab_size=len(vocab), embed_dim=64, dropout=0.4).to(device)
print(ann_model)

ann_model = fit(
    ann_model,
    train_loader,
    val_loader,
    model_name               = "simple_ann",
    n_epochs                 = 30,
    lr                       = 1e-3,
    weight_decay             = 1e-2,
    label_smoothing          = 0.1,
    early_stopping_patience  = 8,
    model_dir                = MODEL_DIR,
    log_dir                  = "runs",
)

SimpleANN(
  (embedding): EmbeddingBag(9002, 64, mode='mean', padding_idx=0)
  (network): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=2, bias=True)
  )
)
[simple_ann] TensorBoard : runs/simple_ann/20260412_140424
               Launch with : tensorboard --logdir runs
Epoch   1/30 | Train: 0.5621 | Val: 0.4350 | Acc: 0.8048 | 1.2s ◀ best
Epoch   2/30 | Train: 0.4584 | Val: 0.3721 | Acc: 0.8332 | 1.1s ◀ best
Epoch   3/30 | Train: 0.4135 | Val: 0.3495 | Acc: 0.8472 | 1.2s ◀ best
Epoch   4/30 | Train: 0.3856 | Val: 0.3411 | Acc: 0.8576 | 1.1s ◀ best
Epoch   5/30 | Train: 0.3625 | Val: 0.3253 | Acc: 0.8636 | 1.1s ◀ best
Epoch   6/30 | Train: 0.3455 | Val: 0.3225 | Acc: 0.8644 | 1.1s ◀ best
Epoch   7/30 | Train: 0.3303 | Val: 0.3225 | Acc: 0.8640 | 1.

In [16]:
_, test_acc, test_preds, test_labels = evaluate(ann_model, test_loader)
print(f"Test Accuracy : {test_acc:.4f}\n")
print(classification_report(test_labels, test_preds, target_names=["Negative", "Positive"]))

Test Accuracy : 0.8600

              precision    recall  f1-score   support

    Negative       0.83      0.81      0.82       989
    Positive       0.88      0.89      0.89      1511

    accuracy                           0.86      2500
   macro avg       0.85      0.85      0.85      2500
weighted avg       0.86      0.86      0.86      2500



## Train Bi-LSTM

In [17]:
lstm_model = BiLSTMClassifier(
    vocab_size    = len(vocab),
    embedding_dim = 128,
    hidden_dim    = 128,
    n_layers      = 1,
    dropout       = 0.4,
).to(device)
print(lstm_model)

lstm_model = fit(
    lstm_model,
    train_loader,
    val_loader,
    model_name               = "bi_lstm",
    n_epochs                 = 30,
    lr                       = 1e-3,
    weight_decay             = 1e-2,
    label_smoothing          = 0.1,
    early_stopping_patience  = 8,
    model_dir                = MODEL_DIR,
    log_dir                  = "runs",
)

BiLSTMClassifier(
  (embedding): Embedding(9002, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=256, out_features=2, bias=True)
)
[bi_lstm] TensorBoard : runs/bi_lstm/20260412_140441
               Launch with : tensorboard --logdir runs
Epoch   1/30 | Train: 0.5045 | Val: 0.3705 | Acc: 0.8448 | 25.5s ◀ best
Epoch   2/30 | Train: 0.3995 | Val: 0.3231 | Acc: 0.8720 | 25.9s ◀ best
Epoch   3/30 | Train: 0.3530 | Val: 0.2938 | Acc: 0.8836 | 25.2s ◀ best
Epoch   4/30 | Train: 0.3148 | Val: 0.2895 | Acc: 0.8840 | 25.8s ◀ best
Epoch   5/30 | Train: 0.2863 | Val: 0.2854 | Acc: 0.8852 | 26.1s ◀ best
Epoch   6/30 | Train: 0.2644 | Val: 0.2985 | Acc: 0.8828 | 25.8s
Epoch   7/30 | Train: 0.2460 | Val: 0.3025 | Acc: 0.8776 | 24.4s
Epoch   8/30 | Train: 0.2359 | Val: 0.3052 | Acc: 0.8796 | 24.2s
Epoch   9/30 | Train: 0.2260 | Val: 0.3248 | Acc: 0.8788 | 22.9s
Epoch  10/30 | Train: 0.2229 | Val: 

In [18]:
print("\nBi-LSTM — Test Set Evaluation")
_, test_acc, test_preds, test_labels = evaluate(lstm_model, test_loader)
print(f"Test Accuracy : {test_acc:.4f}\n")
print(classification_report(test_labels, test_preds, target_names=["Negative", "Positive"]))


Bi-LSTM — Test Set Evaluation
Test Accuracy : 0.8828

              precision    recall  f1-score   support

    Negative       0.86      0.85      0.85       989
    Positive       0.90      0.91      0.90      1511

    accuracy                           0.88      2500
   macro avg       0.88      0.88      0.88      2500
weighted avg       0.88      0.88      0.88      2500



In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 30305), started 0:00:06 ago. (Use '!kill 30305' to kill it.)